In [14]:
import datetime
print(datetime.datetime.now())

2026-08-26 22:39:30.129546


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [4]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [5]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'I’m '
                                                                          'glad '
                                                                          'to '
             

In [8]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem—thanks for letting me know. I’m glad to adjust. Would tomorrow at 2:00 PM or 4:00 PM work for you? If neither fits, tell me a time that does and I’ll adjust.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem—thanks for letting me know. I’m glad to adjust. Would tomorrow at 2:00 PM or 4:00 PM work for you? If neither fits, tell me a time that does and I’ll adjust.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='d05fc95b0d37304fad005eccdb4f50c6')]


In [9]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for letting me know. I’m glad to adjust. Would tomorrow at 2:00 PM or 4:00 PM work for you? If neither fits, tell me a time that does and I’ll adjust.

Best regards,
Seán


## Approve

In [10]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1e11283a-7a59-481d-bdd1-0f56e0a29647'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "**Considering email response tools**\n\nI’m gearing up to help by using the read_email and send_email tools, but there’s a snag. The send_email tool only lets me include a body; it doesn’t ask for recipients or subjects. Since the user wants to read their email and immediately respond within the same thread, I’ll first call read_email to fetch the email content. This will allow me to craft a suitable reply as per the user’s request.**Executing email response process**\n\nI need to fetch the email content first, so I'll use the read_email tool. After retrieving that, I’ll craft a reply

## Reject

In [11]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1e11283a-7a59-481d-bdd1-0f56e0a29647'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "**Considering email response tools**\n\nI’m gearing up to help by using the read_email and send_email tools, but there’s a snag. The send_email tool only lets me include a body; it doesn’t ask for recipients or subjects. Since the user wants to read their email and immediately respond within the same thread, I’ll first call read_email to fetch the email content. This will allow me to craft a suitable reply as per the user’s request.**Executing email response process**\n\nI need to fetch the email content first, so I'll use the read_email tool. After retrieving that, I’ll craft a reply

In [15]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [18]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1e11283a-7a59-481d-bdd1-0f56e0a29647'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "**Considering email response tools**\n\nI’m gearing up to help by using the read_email and send_email tools, but there’s a snag. The send_email tool only lets me include a body; it doesn’t ask for recipients or subjects. Since the user wants to read their email and immediately respond within the same thread, I’ll first call read_email to fetch the email content. This will allow me to craft a suitable reply as per the user’s request.**Executing email response process**\n\nI need to fetch the email content first, so I'll use the read_email tool. After retrieving that, I’ll craft a reply